In [1]:
import pandas as pd 
import numpy as np

In [2]:
df=pd.read_csv("housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [3]:
df["income_cat"]=pd.cut(df["median_income"],bins=[0,1.5,2.5,3.5,4.5,5.5,6,np.inf],labels=[1,2,3,4,5,6,7])

In [4]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,7
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,7
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,7
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,6
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,4


In [5]:
from sklearn.model_selection import StratifiedShuffleSplit # can do multiple splits

In [6]:
sss=StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_idx,test_idx in sss.split(df,df["income_cat"]):
    train_set=df.iloc[train_idx]
    test_set=df.iloc[test_idx]

In [7]:
#droping "income_cat" column now

In [8]:
df=train_set.copy()

In [9]:
df.drop("income_cat",axis=1,inplace=True)

In [10]:
#Before Handling missing data: remove the main feature
# here it is "median_house_value"

In [11]:
housing=df.drop("median_house_value",axis=1)
housing_label=df["median_house_value"].copy()

In [12]:
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
17751,-121.84,37.33,26.0,1934.0,408.0,2059.0,416.0,3.6765,<1H OCEAN
12609,-121.49,38.51,30.0,3166.0,607.0,1857.0,579.0,3.1768,INLAND
1030,-120.25,38.55,15.0,4403.0,891.0,1103.0,433.0,3.0125,INLAND
9844,-121.90,36.59,42.0,2689.0,510.0,1023.0,459.0,4.6182,NEAR OCEAN
1253,-122.01,39.21,50.0,1592.0,372.0,781.0,307.0,2.2679,INLAND


In [13]:
#now lets handle missing value
housing.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        162
population              0
households              0
median_income           0
ocean_proximity         0
dtype: int64

In [14]:
num_att=housing.drop("ocean_proximity",axis=1).copy().columns.tolist()
cat_att=["ocean_proximity"]

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
num_pipeline=Pipeline([
    ("impute",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])
cat_pipeline=Pipeline([
    ("Onehot",OneHotEncoder())
])
full_pipeline=ColumnTransformer([
    ("num",num_pipeline,num_att),
    ("cat",cat_pipeline,cat_att)
])
housing_prepared=full_pipeline.fit_transform(housing)

In [16]:
housing_df=pd.DataFrame(housing_prepared,columns=full_pipeline.get_feature_names_out(),index=housing.index)
housing_df

,num__longitude,num__latitude,num__housing_median_age,num__total_rooms,num__total_bedrooms,num__population,num__households,num__median_income,cat__ocean_proximity_<1H OCEAN,cat__ocean_proximity_INLAND,cat__ocean_proximity_ISLAND,cat__ocean_proximity_NEAR BAY,cat__ocean_proximity_NEAR OCEAN
17751,-1.125710,0.790703,-0.208355,-0.319885,-0.306802,0.545493,-0.218651,-0.103385,1.0,0.0,0.0,0.0,0.0
12609,-0.951183,1.343558,0.108582,0.234927,0.159554,0.370543,0.200686,-0.365714,0.0,1.0,0.0,0.0,0.0
1030,-0.332857,1.362299,-1.079932,0.791990,0.825107,-0.282490,-0.174916,-0.451967,0.0,1.0,0.0,0.0,0.0
9844,-1.155629,0.443998,1.059394,0.020118,-0.067765,-0.351777,-0.108028,0.390982,0.0,0.0,0.0,0.0,1.0
1253,-1.210481,1.671522,1.693268,-0.473899,-0.391168,-0.561371,-0.499066,-0.842861,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19593,-0.642020,0.917204,0.267051,0.871249,0.879008,1.137033,0.913301,-0.636022,0.0,1.0,0.0,0.0,0.0
2676,1.990850,-1.256732,-0.921463,-0.812100,-0.920798,-0.828127,-0.892676,-0.354637,0.0,1.0,0.0,0.0,0.0
12392,1.581957,-0.891286,-0.921463,0.740652,0.829794,-0.408073,-0.156908,-0.738916,0.0,1.0,0.0,0.0,0.0
6127,0.819023,-0.722619,0.187817,-0.632867,-0.667701,-0.433189,-0.661141,-0.268226,1.0,0.0,0.0,0.0,0.0


In [17]:
#training 
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

lr=LinearRegression()
dtr=DecisionTreeRegressor()
rfr=RandomForestRegressor()

u]
